# DPP Follow-up Analysis — Colleague's Questions (2026-06)

Companion to `dpp_analysis.ipynb`. Self-contained: reloads all sources and answers the
specific follow-up questions raised after the *DPP Crosswalk Analysis — 2026-06-04* and
*State of Affairs* briefings.

Sections:
- **Q-A** — What ballpark of Pokorny data are we missing? (reconciling ~120 / ~700 / 20% / 30%)
- **Q-B** — Are we missing *reflexes* (not just roots)?
- **Q-F** — Column-level crosswalk semantics DPP ↔ IELex ↔ PIET *(to come)*

Every number printed here is reproduced live from the data — nothing is hand-typed.

In [68]:
import re
from pathlib import Path
import pandas as pd

REPO = Path("../../../")  # src/ielex/notebooks/ → repo root
DPP_FILE = REPO / "data/ielex/derbi_pie_pokorny/pok_all(Sheet1)_utf8.csv"
IELEX_DIR = REPO / "data/ielex/current_ielex"

dpp          = pd.read_csv(DPP_FILE, encoding="utf-8-sig", low_memory=False)
etyma        = pd.read_csv(IELEX_DIR / "lex_etyma.csv")
etyma_reflex = pd.read_csv(IELEX_DIR / "lex_etyma_reflex.csv")
reflex       = pd.read_csv(IELEX_DIR / "lex_reflex.csv")
language     = pd.read_csv(IELEX_DIR / "lex_language.csv")

print(f"DPP rows:            {len(dpp):>7,}")
print(f"IELex etyma:         {len(etyma):>7,}")
print(f"IELex reflexes:      {len(reflex):>7,}")
print(f"Etymon-reflex links: {len(etyma_reflex):>7,}")

DPP rows:             67,656
IELex etyma:           2,222
IELex reflexes:       63,116
Etymon-reflex links:  64,301


In [69]:
def norm_strip(s: str) -> str:
    """Lowercase, collapse whitespace, AND strip a leading ordinal ('1. ', '2. ').

    Pokorny numbers his homophones (1. ad-, 2. ad-). DPP keeps that number INSIDE
    the root string; IELex stores the SAME number in its separate `homograph_number`
    column, with entry = 'ad-'. So a bare string match fails ('1. ad-' != 'ad-').
    Stripping the ordinal lets the strings match, but DISCARDS the homophone id --
    for reflex attribution you must re-join DPP ordinal <-> IELex homograph_number
    (verified ~1:1 in section A.2b), not match on the stripped string alone.
    """
    if not isinstance(s, str):
        return ""
    s = re.sub(r"^\d+\.\s*", "", s.strip()).strip().lower()
    return re.sub(r"\s+", " ", s)


def norm_plain(s: str) -> str:
    """Lowercase + collapse whitespace only (NO ordinal strip) — the naive matcher."""
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s.strip().lower())


def ordinal_of(s: str):
    """Return Pokorny's leading homophone number from a DPP root string, or None."""
    m = re.match(r"^(\d+)\.", s.strip()) if isinstance(s, str) else None
    return m.group(1) if m else None

## Q-A — What ballpark are we in?

### A.1 — Empty roots: how many IELex etyma have zero reflexes?

**Data model (pincushion).** Roots and daughter-words are joined by a junction table:

```
lex_etyma (2,222)          lex_etyma_reflex (64,301)        lex_reflex (63,116)
  id  ◄──────────────────── etyma_id   reflex_id ───────────► id
  entry, homograph_number,  (the "edges")                     entries (JSON surface
  page_number, gloss,                                          forms), gloss,
  lexicon_id                                                   language_id ─► lex_language
```

An etymon is **empty** ⟺ its `id` never appears as an `etyma_id` in `lex_etyma_reflex`.
Equivalent SQL:

```sql
SELECT COUNT(*) FROM lex_etyma e
LEFT JOIN lex_etyma_reflex er ON er.etyma_id = e.id
WHERE er.etyma_id IS NULL;   -- 681
```

The cell first runs three integrity checks so the count isn't resting on an assumption:
no orphan FKs, a single lexicon (empties aren't hidden in a 2nd lexicon), and `entries`
is a *surface-form* field, not an alternate etymon link. `lex_etyma_reflex` is the
**sole** etymon↔reflex path (and it is many-to-many: 64,301 links > 63,116 reflexes).

In [70]:
# --- Integrity checks: confirm lex_etyma_reflex is the sole, clean etymon<->reflex link ---
eids = set(etyma["id"])
rids = set(reflex["id"])
linked_eids = set(etyma_reflex["etyma_id"])
linked_rids = set(etyma_reflex["reflex_id"])

orphan_e = linked_eids - eids          # etyma_id pointing at no etymon
orphan_r = linked_rids - rids          # reflex_id pointing at no reflex
lexicons = etyma["lexicon_id"].unique()
entries_nonempty = reflex["entries"].fillna("").str.strip().ne("").sum()

assert not orphan_e, f"orphan etyma_id in link table: {len(orphan_e)}"
assert not orphan_r, f"orphan reflex_id in link table: {len(orphan_r)}"
print("Integrity checks")
print(f"  orphan etyma_id / reflex_id in link table : {len(orphan_e)} / {len(orphan_r)}")
print(f"  distinct lexicon_id among etyma           : {sorted(lexicons)}  (single -> no hidden lexicon)")
print(f"  lex_reflex.entries populated (surface form, not a link) : {entries_nonempty:,}/{len(reflex):,}")
print(f"  link table is many-to-many                : {len(etyma_reflex):,} links > {len(reflex):,} reflexes")

# --- The count itself ---
linked = linked_eids
etyma["has_reflex"] = etyma["id"].isin(linked)

n_total = len(etyma)
n_with  = int(etyma["has_reflex"].sum())
n_empty = n_total - n_with

print("\nEmpty-root count (etyma with no row in lex_etyma_reflex)")
print(f"  IELex etyma total         : {n_total:,}")
print(f"    with >=1 reflex         : {n_with:,}")
print(f"    with 0 reflexes (empty) : {n_empty:,}  ({n_empty/n_total*100:.1f}%)   <- the real gap")

Integrity checks
  orphan etyma_id / reflex_id in link table : 0 / 0
  distinct lexicon_id among etyma           : [np.int64(1)]  (single -> no hidden lexicon)
  lex_reflex.entries populated (surface form, not a link) : 63,116/63,116
  link table is many-to-many                : 64,301 links > 63,116 reflexes

Empty-root count (etyma with no row in lex_etyma_reflex)
  IELex etyma total         : 2,222
    with >=1 reflex         : 1,541
    with 0 reflexes (empty) : 681  (30.6%)   <- the real gap


### A.2 — DPP root inventory, alignment, and what DPP can fill

In [71]:
# DPP unique roots, raw and ordinal-stripped
dpp_roots = dpp.drop_duplicates("root")[["root"]].copy()
dpp_roots["stripped"] = dpp_roots["root"].apply(norm_strip)
dpp_stripped = set(dpp_roots["stripped"])

# Stripped normalization of IELex entries
etyma["entry_stripped"] = etyma["entry"].apply(norm_strip)

# Alignment: exact stripped match OR IELex entry starts with a DPP root (prefix)
def aligned(entry: str) -> bool:
    if entry in dpp_stripped:
        return True
    return any(r and entry.startswith(r) for r in dpp_stripped)

etyma["dpp_aligned"] = etyma["entry_stripped"].apply(aligned)
n_aligned = int(etyma["dpp_aligned"].sum())

print(f"DPP unique roots (raw)      : {len(dpp_roots):,}")
print(f"DPP unique roots (stripped) : {len(dpp_stripped):,}  (homophone variants collapse)")
print(f"IELex etyma aligned to DPP  : {n_aligned:,}  ({n_aligned/n_total*100:.0f}%)")
print(f"IELex etyma NOT aligned     : {n_total - n_aligned:,}")

gap_fillable = int((~etyma["has_reflex"] & etyma["dpp_aligned"]).sum())
print(f"\nEmpty roots fillable from DPP now : {gap_fillable:,}  (of {n_empty:,} empty)")

DPP unique roots (raw)      : 1,518
DPP unique roots (stripped) : 1,400  (homophone variants collapse)
IELex etyma aligned to DPP  : 1,503  (68%)
IELex etyma NOT aligned     : 719

Empty roots fillable from DPP now : 112  (of 681 empty)


### A.2b — Homograph encoding: DPP ordinal **==** IELex `homograph_number`

Important correction to an early assumption. It is *not* true that "IELex has one entry
`ad-` while DPP has `1. ad-`, `2. ad-`." **Both** datasets distinguish Pokorny's
homophones, they just encode the number differently:

| Source | `ad-` homophone 1 | `ad-` homophone 2 | where the number lives |
|--------|-------------------|-------------------|------------------------|
| DPP    | `root = "1. ad-"` | `root = "2. ad-"` | inside the root string |
| IELex  | `entry="ad-", homograph_number=1` | `entry="ad-", homograph_number=2` | separate column |

The cell below verifies the numbers correspond, so the right join key is
`(stripped_root, ordinal)` ↔ `(entry, homograph_number)`.

In [72]:
from collections import defaultdict

# DPP: stripped root -> set of in-string ordinals
dpp_ord = defaultdict(set)
for root in dpp["root"]:
    dpp_ord[norm_strip(root)].add(ordinal_of(root))

# IELex: stripped entry -> set of homograph_numbers (string, NULL/blank dropped)
def clean_hg(v):
    v = str(v).strip()
    return None if v in ("", "NULL", "nan") else v

iel_hg = defaultdict(set)
for _, r in etyma.iterrows():
    iel_hg[r["entry_stripped"]].add(clean_hg(r["homograph_number"]))

# Roots present in both, with >1 homophone on either side
multi = [k for k in dpp_ord
         if k in iel_hg
         and (len({x for x in dpp_ord[k] if x}) > 1 or len({x for x in iel_hg[k] if x}) > 1)]
subset = sum(1 for k in multi
             if {x for x in dpp_ord[k] if x}
             and {x for x in dpp_ord[k] if x} <= {x for x in iel_hg[k] if x})

print(f"Multi-homophone roots (in both)             : {len(multi)}")
print(f"  DPP ordinals are a SUBSET of IELex homog. : {subset}/{len(multi)}")
print("=> DPP's in-string ordinal == Pokorny's homophone number == IELex homograph_number.")
print("   Correct join key: (stripped_root, ordinal) <-> (entry, homograph_number).")

# Worked example
print("\nExample 'ad-':")
for root in sorted({r for r in dpp["root"] if norm_strip(r) == "ad-"}):
    print(f"  DPP    {root!r}")
print(etyma.loc[etyma["entry_stripped"] == "ad-", ["entry", "homograph_number", "gloss"]]
      .to_string(index=False))

Multi-homophone roots (in both)             : 123
  DPP ordinals are a SUBSET of IELex homog. : 122/123
=> DPP's in-string ordinal == Pokorny's homophone number == IELex homograph_number.
   Correct join key: (stripped_root, ordinal) <-> (entry, homograph_number).

Example 'ad-':
  DPP    '1. ad-'
  DPP    '2. ad-'
entry homograph_number                         gloss
  ad-                1   {"en":"<b>at<\/b>, by, to"}
  ad-                2 {"en":"to fix, put in order"}


### A.3 — The three different "~700" numbers people conflate

"~700–800 missing" is ambiguous because **three** unrelated counts all land near 700.

In [73]:
empty_roots = n_empty  # (1) IELex roots with 0 reflexes

# (2) DPP roots with NO naive match (result of ordinal splitting)
ielex_plain = set(etyma["entry"].apply(norm_plain))
dpp_plain   = set(dpp.drop_duplicates("root")["root"].apply(norm_plain))
naive_unmatched = len(dpp_plain - ielex_plain)

# (2b) ...the genuine figure after the ordinal-strip fix
ielex_strip_set   = set(etyma["entry_stripped"])
genuine_unmatched = len(dpp_stripped - ielex_strip_set)

# (3) IELex roots not aligned to any DPP root (the inverse direction)
ielex_unaligned = n_total - n_aligned

print("Three different '~700' numbers that get conflated:")
print(f"  (1) IELex roots with 0 reflexes        : {empty_roots:,}")
print(f"  (2) DPP roots w/ no NAIVE IELex match   : {naive_unmatched:,}  <- ordinal-split result")
print(f"      -> after ordinal-strip fix          : {genuine_unmatched:,}  (the real number)")
print(f"  (3) IELex roots not covered by DPP      : {ielex_unaligned:,}")

Three different '~700' numbers that get conflated:
  (1) IELex roots with 0 reflexes        : 681
  (2) DPP roots w/ no NAIVE IELex match   : 702  <- ordinal-split result
      -> after ordinal-strip fix          : 147  (the real number)
  (3) IELex roots not covered by DPP      : 719


### A.4 — Do our roots span the whole Pokorny book?

In [74]:
# page_number includes ranges like '918-19'; take the first page of each
first_page = (
    etyma["page_number"].astype(str).str.extract(r"(\d+)")[0].astype("Float64")
)
covered = first_page.dropna()
print(f"Etyma with a page reference : {covered.notna().sum():,} / {n_total:,}")
print(f"Pokorny pages covered       : {int(covered.min())}-{int(covered.max())}")
print(f"Distinct book pages touched : {covered.nunique():,}")
print(f"Roots per covered page      : {len(covered)/covered.nunique():.2f}")

Etyma with a page reference : 2,222 / 2,222
Pokorny pages covered       : 1-1183
Distinct book pages touched : 930
Roots per covered page      : 2.39


### A.5 — Reconciliation summary

In [75]:
summary = pd.DataFrame([
    ("IELex roots (etyma)",             n_total,           "We already have these"),
    ("DPP/Starling unique roots (raw)", len(dpp_roots),    "Fewer than IELex - NOT a superset"),
    ("IELex roots with 0 reflexes",     n_empty,           f"{n_empty/n_total*100:.1f}% - the real gap is reflexes"),
    ("  ...fillable from DPP now",       gap_fillable,      "Immediate gap-fill target (~the '120')"),
    ("DPP roots not in IELex (fixed)",   genuine_unmatched, "Candidate net-new roots"),
    ("IELex roots not covered by DPP",   ielex_unaligned,   "DPP can't help these"),
], columns=["metric", "count", "interpretation"])
summary

,metric,count,interpretation
0,IELex roots (etyma),2222,We already have these
1,DPP/Starling unique roots (raw),1518,Fewer than IELex - NOT a superset
2,IELex roots with 0 reflexes,681,30.6% - the real gap is reflexes
3,...fillable from DPP now,112,Immediate gap-fill target (~the '120')
4,DPP roots not in IELex (fixed),147,Candidate net-new roots
5,IELex roots not covered by DPP,719,DPP can't help these


## Q-B — Are IELex roots *under-populated*? (reflex-level gap)

**Colleague's question:** even for roots we do cover, do we have all the reflexes?

**Scope.** We compare, root-by-root, the number of daughter-word entries in IELex against the
number in DPP.  Because both sources derive from the same Pokorny 1959 text, a systematic
shortfall in IELex is a genuine gap, not a source difference.

**Method agreed before computing:**
- Join key: `(norm_strip(dpp.root), ordinal)` ↔ `(etyma.entry_stripped, hg_num_prefix)` —
  homograph-aware, matching IELex alpha sub-homographs (3a, 3b…) to their numeric prefix.
- **IELex count unit:** `COUNT(DISTINCT reflex_id)` per etyma group from `lex_etyma_reflex`.
- **DPP count unit:** non-blank `reflex` rows per root group (row = one reflex; `pok_wd_index`
  is unique per row).  A dedup lower bound (distinct strings) is also reported.
- **Footnote:** 686 DPP pok_all_index groups have blank `root` for all rows (6,676 rows,
  ~10% of DPP); these use only `web_root` and cannot be reliably joined — excluded from core
  analysis.  The reported gap is therefore a *lower bound*.

> **Interpretation refined (2026-06-07, see §Q-G).** These counts stand, but the 43.4% of roots where *IELex ≥ DPP* is **not** richer Pokorny — it is Webster's-7th / AHD English padding. Only 1.7% of IELex reflexes cite Pokorny; the IELex-only surplus over DPP is 62.7% Webster's + 27.0% AHD, 0.2% Pokorny. On a Pokorny-only basis the reflex gap is *larger*, not smaller.

### B.1 — IELex: distinct-reflex count per etymon group

Group IELex etyma by `(entry_stripped, hg_key)` where `hg_key` is the *numeric prefix* of
`homograph_number` (so `3a`, `3b`, `3c` all collapse to `"3"`, matching DPP's digit-only
ordinal).  Sum distinct-reflex counts across sub-variants within each group.

In [76]:
import re as _re, json as _json
from collections import defaultdict
import pandas as pd

SENTINEL = "__none__"  # used for entries with no ordinal / no homograph_number

def hg_num_prefix(v):
    """Strip alpha suffix: '3a' -> '3', '3' -> '3', blank/NULL -> None."""
    if pd.isna(v) or str(v).strip() in ("", "NULL", "nan"): return None
    return _re.sub(r"[a-z]+$", "", str(v).strip()) or None

# lex_etyma_reflex is many-to-many (64,301 links > 63,116 reflexes).
# Count DISTINCT reflex_id per etyma_id — same integrity standard as Q-A.
iel_rcount = (
    etyma_reflex.groupby("etyma_id")["reflex_id"].nunique()
    .rename("iel_rc")
)

# entry_stripped already computed in Q-A; add hg columns
etyma["hg_prefix"] = etyma["homograph_number"].apply(hg_num_prefix)
etyma["hg_key"]    = etyma["hg_prefix"].fillna(SENTINEL)
etyma["iel_count"] = etyma["id"].map(iel_rcount).fillna(0).astype(int)

# Group by (entry_stripped, hg_key): sum counts across alpha sub-homographs
etyma_g = (
    etyma.groupby(["entry_stripped", "hg_key"], dropna=False)
    .agg(iel_total=("iel_count","sum"),
         etyma_ids=("id",list),
         entry=("entry","first"),
         hg_prefix=("hg_prefix","first"))
    .reset_index()
)

alpha_collapsed = int(
    (etyma["homograph_number"].apply(lambda x: bool(
        isinstance(x, str) and _re.search(r"[a-z]", x)
    ))).sum()
)
print(f"IELex etyma groups after collapsing alpha sub-homographs: {len(etyma_g):,}")
print(f"  (from {len(etyma):,} etyma; {alpha_collapsed} had alpha sub-homograph numbers)") 

IELex etyma groups after collapsing alpha sub-homographs: 2,219
  (from 2,222 etyma; 18 had alpha sub-homograph numbers)


### B.2 — DPP: count reflex rows per root group

`root` is blank for ~10% of DPP rows (overflow / notes rows).  Forward-fill `root` within
each `pok_all_index` block (block id is always filled and constant per entry) so overflow rows
inherit their parent root.  Extract ordinal and stripped form from the filled root.

In [77]:
dpp_s = dpp.sort_values(["pok_all_index","pok_wd_index"]).copy()
# ffill then bfill in case the very first rows of a block have blank root (rare)
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    dpp_s["root_filled"] = (
        dpp_s.groupby("pok_all_index")["root"]
        .transform(lambda x: x.ffill().bfill())
    )

n_blank_rows   = dpp_s["root_filled"].isna().sum()
n_blank_groups = dpp_s[dpp_s["root_filled"].isna()]["pok_all_index"].nunique()

dpp_s["stripped"] = dpp_s["root_filled"].apply(norm_strip)
dpp_s["ordinal"]  = dpp_s["root_filled"].apply(ordinal_of)
dpp_s["ord_key"]  = dpp_s["ordinal"].fillna(SENTINEL)

# normalize abbr: strip whitespace and lowercase (raw abbr has whitespace variants)
dpp_s["abbr_norm"] = dpp_s["abbr"].fillna("").str.strip().str.lower()
dpp_s["abbr_norm"] = dpp_s["abbr_norm"].str.replace(r"\s+", " ", regex=True)

# Count only rows with a non-blank reflex (actual daughter-word entries)
dpp_valid = dpp_s[dpp_s["reflex"].notna() & (dpp_s["reflex"].str.strip() != "")].copy()

dpp_counts = (
    dpp_valid.groupby(["stripped","ord_key"], dropna=False)
    .agg(dpp_rows=("reflex","count"), dpp_distinct=("reflex","nunique"))
    .reset_index()
)

print(f"DPP root groups (from `root` column): {len(dpp_counts):,}")
print(f"  Footnote: {n_blank_groups:,} pok_all_index groups have blank `root` for ALL rows")
print(f"  ({n_blank_rows:,} rows excluded — use web_root-based join to recover, lower bound)") 

DPP root groups (from `root` column): 1,510
  Footnote: 686 pok_all_index groups have blank `root` for ALL rows
  (6,676 rows excluded — use web_root-based join to recover, lower bound)


### B.3 — Homograph-aware join, delta, summary

Inner join on `(stripped, ord_key)` = `(entry_stripped, hg_key)`.  For each matched root
group, δ = DPP_rows − IELex_count.  Roots with δ > 0 are *under-populated*.

In [78]:
merged = pd.merge(
    dpp_counts, etyma_g,
    left_on=["stripped","ord_key"],
    right_on=["entry_stripped","hg_key"],
    how="inner"
)

merged["delta"]          = merged["dpp_rows"] - merged["iel_total"]
merged["delta_distinct"] = merged["dpp_distinct"] - merged["iel_total"]

populated   = merged[merged["iel_total"] > 0].copy()
under_pop   = populated[populated["delta"] > 0].copy()
over_pop    = populated[populated["delta"] < 0].copy()
at_parity   = populated[populated["delta"] == 0].copy()
empty_match = merged[merged["iel_total"] == 0].copy()

n_pop = len(populated)

print("=== Q-B: Reflex-level coverage ===")
print(f"Matched root groups (inner join, homograph-aware): {len(merged):,}")
print(f"  of which IELex empty (Q-A gap):                  {len(empty_match):,}")
print(f"  IELex has >=1 reflex:                            {n_pop:,}")
print()
print(f"Among non-empty matched roots ({n_pop:,}):")
print(f"  Under-populated  (DPP > IELex)  : {len(under_pop):,}  ({len(under_pop)/n_pop*100:.1f}%)")
print(f"  At parity        (DPP == IELex) : {len(at_parity):,}  ({len(at_parity)/n_pop*100:.1f}%)")
print(f"  IELex >= DPP                    : {len(over_pop):,}  ({len(over_pop)/n_pop*100:.1f}%)")
print()
print(f"Total reflex shortfall (sum δ where δ>0):")
print(f"  DPP row count method  : {int(under_pop['delta'].sum()):>6,}")
print(f"  DPP distinct strings  : {int(under_pop['delta_distinct'].sum()):>6,}  (lower bound)")
print()
print("Top 20 most under-populated roots (DPP rows vs IELex distinct reflexes):")
cols = ["entry","hg_prefix","dpp_rows","iel_total","delta"]
print(under_pop.nlargest(20,"delta")[cols].to_string(index=False)) 

=== Q-B: Reflex-level coverage ===
Matched root groups (inner join, homograph-aware): 1,358
  of which IELex empty (Q-A gap):                  1
  IELex has >=1 reflex:                            1,357

Among non-empty matched roots (1,357):
  Under-populated  (DPP > IELex)  : 738  (54.4%)
  At parity        (DPP == IELex) : 30  (2.2%)
  IELex >= DPP                    : 589  (43.4%)

Total reflex shortfall (sum δ where δ>0):
  DPP row count method  : 14,743
  DPP distinct strings  : 13,882  (lower bound)

Top 20 most under-populated roots (DPP rows vs IELex distinct reflexes):
                                                                         entry hg_prefix  dpp_rows  iel_total  delta
                                                                         u̯er-         3       454         72    382
                                                                          gel-         1       470        201    269
                                                               g

### B.4 — Per-language gap: which languages are absent in IELex?

Map DPP German language abbrs (IEW standard) to IELex language_ids.  For each matched root
group, find which DPP languages have zero corresponding reflexes in IELex.

**Dict coverage:** 157 DPP abbr → IELex language mappings; covers 92.7% of DPP valid reflex
rows.  Remaining 7.3% are compound abbrs (`ahd. as.`), dialect qualifiers
(`norw. mdartl.`, `nhd. dial.`), and low-frequency forms — flagged as unmapped.

In [79]:
# DPP German IEW abbr -> IELex language abbr (157 mappings, 92.7% row coverage)
DPP_TO_IELEX_ABBR = {
    # Greek
    "gr.": "Gk",
    "aeol.": "Aeol",
    "äol.": "Aeol",
    "dor.": "Dor",
    "ion.": "Ion",
    "att.": "Att",
    "hom.": "Hom",
    "myken.": "Myc",
    # Indic / Sanskrit
    "ai.": "Skt",
    "aind.": "Skt",
    "skr.": "Skt",
    "ved.": "Ved",
    "pali": "Pali",
    "pali.": "Pali",
    "prakr.": "Prak",
    # Latin / Italic
    "lat.": "Lat",
    "alat.": "OLat",
    "osk.": "Osc",
    "umbr.": "Umb",
    "fal.": "Fal",
    "mlat.": "MLat",
    "vlat.": "VLat",
    "kirchenlat.": "MLat",
    # Celtic
    "air.": "OIr",
    "mir.": "MIr",
    "ir.": "Ir",
    "nir.": "Ir",
    "kymr.": "W",
    "cymr.": "W",
    "ncymr.": "W",
    "nkymr.": "W",
    "mkymr.": "MW",
    "mcymr.": "MW",
    "acymr.": "OW",
    "akymr.": "OW",
    "bret.": "Bret",
    "abret.": "OBret",
    "nbret.": "Bret",
    "mbreton.": "MBret",
    "mbret.": "MBret",
    "corn.": "Corn",
    "akorn.": "OCorn",
    "acorn.": "OCorn",
    "mkorn.": "MCorn",
    "gall.": "Gaul",
    "gaul.": "Gaul",
    "kelt.": "Celt",
    # Germanic
    "got.": "Go",
    "aisl.": "OIce",
    "anord.": "ON",
    "an.": "ON",
    "isl.": "Ice",
    "nisl.": "Ice",
    "aschw.": "OSw",
    "aschwed.": "OSw",
    "nschwed.": "Sw",
    "schwed.": "Sw",
    "schwed. dial.": "Sw",
    "adän.": "ODan",
    "ndän.": "Dan",
    "dän.": "Dan",
    "anorw.": "ONorw",
    "nnorw.": "Norw",
    "norw.": "Norw",
    "norw. dial.": "Norw",
    "norw. mdartl.": "Norw",
    "ags.": "AS",
    "ae.": "OE",
    "me.": "ME",
    "mengl.": "ME",
    "engl.": "NE",
    "ne.": "NE",
    "as.": "OS",
    "ahd.": "OHG",
    "mhd.": "MHG",
    "nhd.": "NHG",
    "nhd. dial.": "NHG",
    "mnl.": "MDu",
    "mndl.": "MDu",
    "nnl.": "Du",
    "ndl.": "Du",
    "nl.": "Du",
    "nd.": "LG",
    "ndd.": "LG",
    "afries.": "OFris",
    "mfries.": "MFris",
    "nfries.": "NFris",
    "mnd.": "MLG",
    "lang.": "Lang",
    "schweiz.": "Swiss",
    # Baltic
    "lit.": "Lith",
    "alit.": "OLith",
    "lett.": "Latv",
    "apr.": "OPrus",
    # Slavic
    "aksl.": "OCS",
    "abg.": "OCS",
    "ksl.": "OCS",
    "russ.-ksl.": "OCS",
    "russ.": "Russ",
    "russ. dial.": "Russ",
    "bulg.": "Bulg",
    "serb.": "Serb",
    "kroat.": "Croat",
    "poln.": "Pol",
    "sorb.": "Sorb",
    "tschech.": "Cz",
    "čech.": "Cz",
    "slow.": "Slovene",
    "sloven.": "Slovene",
    "slovak.": "Slovak",
    "klr.": "Ukr",
    # Iranian
    "av.": "Av",
    "aw.": "Av",
    "jav.": "YAv",
    "aav.": "OAv",
    "apers.": "OPers",
    "mpers.": "MPers",
    "npers.": "NPers",
    "np.": "NPers",
    "oss.": "Oss",
    "pahlavi.": "Pahl",
    "sogd.": "Sogd",
    "kurd.": "Kurd",
    "baktrian.": "Bact",
    "iran.": "Iran",
    # Armenian
    "arm.": "Arm",
    # Tocharian
    "toch.": "Toch",
    "toch. a": "TochA",
    "toch. b": "TochB",
    "toch. a/b": "Toch",
    "toch.a": "TochA",
    "toch.b": "TochB",
    "toch. ab": "Toch",
    # Anatolian
    "heth.": "Hitt",
    "hitt.": "Hitt",
    "luw.": "Luw",
    "lyc.": "Lyc",
    "lyd.": "Lyd",
    # Albanian
    "alb.": "Alb",
    # Phrygian / Thracian / Illyrian / Messapic
    "phryg.": "Phryg",
    "thrak.": "Thrac",
    "illyr.": "Illyr",
    "messap.": "Mess",
    # Venetic
    "venez.": "Ven",
    # Romance
    "afz.": "OF",
    "afrz.": "OF",
    "mfrz.": "MFr",
    "frz.": "Fr",
    "nfr.": "Fr",
    "fr.": "Fr",
    "prov.": "Prov",
    "aprov.": "OProv",
    "span.": "Sp",
    "aspan.": "OSp",
    "port.": "Port",
    "ital.": "It",
    "rum.": "Rum",
}

lang_abbr2id = dict(zip(language["abbr"], language["id"]))
lid2name = {row["id"]: _json.loads(row["name"])["en"] for _, row in language.iterrows()}


def dpp_abbr_to_lid(abbr_norm):
    ielex_abbr = DPP_TO_IELEX_ABBR.get(abbr_norm)
    return lang_abbr2id.get(ielex_abbr) if ielex_abbr else None


# per-root IELex language set: etyma_id -> set of language_ids
rid2lid_map = dict(zip(reflex["id"], reflex["language_id"]))
eid2langs = defaultdict(set)
for _, row in etyma_reflex.iterrows():
    lid = rid2lid_map.get(row["reflex_id"])
    if lid is not None:
        eid2langs[int(row["etyma_id"])].add(int(lid))

# for each matched root: which DPP languages appear but are absent in IELex?
gap_records = []
for _, mrow in merged.iterrows():
    eids = mrow["etyma_ids"]
    iel_langs = set()
    for eid in eids:
        iel_langs |= eid2langs.get(int(eid), set())

    mask_root = (dpp_valid["stripped"] == mrow["stripped"]) & (
        dpp_valid["ord_key"] == mrow["ord_key"]
    )
    for abbr in dpp_valid.loc[mask_root, "abbr_norm"].dropna().unique():
        lid = dpp_abbr_to_lid(abbr)
        gap_records.append(
            {
                "dpp_abbr": abbr,
                "iel_lang_id": lid,
                "absent": (lid not in iel_langs) if lid is not None else None,
                "mapped": lid is not None,
                "entry": mrow["entry"],
                "hg_prefix": mrow["hg_prefix"],
            }
        )

gap_df = pd.DataFrame(gap_records)
mapped_n = gap_df["mapped"].sum()
unmapped_n = (~gap_df["mapped"]).sum()

print(f"DPP (abbr × root) instances in matched roots : {len(gap_df):,}")
print(
    f"  Mapped to IELex language_id : {mapped_n:,}  ({mapped_n/len(gap_df)*100:.1f}%)"
)
print(
    f"  Unmapped (compound/dialect)  : {unmapped_n:,}  ({unmapped_n/len(gap_df)*100:.1f}%)"
)

# summarize absences
absent_df = gap_df[gap_df["mapped"] & (gap_df["absent"] == True)]
lang_absent = (
    absent_df.groupby(["iel_lang_id", "dpp_abbr"])
    .size()
    .rename("absent_root_count")
    .reset_index()
    .sort_values("absent_root_count", ascending=False)
)
lang_absent["lang_name"] = lang_absent["iel_lang_id"].map(lid2name)

print()
print("Top 20 languages most often absent in IELex (by distinct root count):")
# de-dup on iel_lang_id (a lang may appear under >1 DPP abbr)
top_by_lang = (
    absent_df.groupby("iel_lang_id")
    .apply(lambda x: x["entry"].nunique())
    .rename("roots_absent")
    .reset_index()
    .sort_values("roots_absent", ascending=False)
    .head(20)
)
top_by_lang["lang_name"] = top_by_lang["iel_lang_id"].map(lid2name)
top_by_lang["dpp_abbrs"] = top_by_lang["iel_lang_id"].apply(
    lambda lid: ", ".join(
        sorted(absent_df[absent_df["iel_lang_id"] == lid]["dpp_abbr"].unique())
    )
)
print(top_by_lang.to_string(index=False))

roots_with_gap = absent_df["entry"].nunique()
print(
    f"\nMatched roots with >=1 mapped-language gap: {roots_with_gap:,} of {len(merged):,}"
)
print(f"Top 15 unmapped DPP abbrs:")
print(gap_df[~gap_df["mapped"]]["dpp_abbr"].value_counts().head(15).to_string())

DPP (abbr × root) instances in matched roots : 21,109
  Mapped to IELex language_id : 17,907  (84.8%)
  Unmapped (compound/dialect)  : 3,202  (15.2%)

Top 20 languages most often absent in IELex (by distinct root count):
 iel_lang_id  roots_absent           lang_name                                 dpp_abbrs
       400.0           808         Anglo-Saxon                                      ags.
       456.0           469       Old Icelandic                                     aisl.
       443.0           451     New High German                          nhd., nhd. dial.
       551.0           378          Lithuanian                                      lit.
       718.0           368            Sanskrit                                 ai., skr.
       552.0           368             Latvian                                     lett.
       391.0           351               Welsh                      cymr., kymr., ncymr.
       575.0           313 Old Church Slavonic             abg., ak

/var/folders/qp/sgyn9rjs7jl_zv5cksz_txj40000gn/T/ipykernel_67247/1671679653.py:246: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x["entry"].nunique())


In [80]:
# --- B.5 Sanity checks: eyeball ā and ad- ---
print("=== Sanity check: ā ===")
a_mask = etyma["entry"] == "ā"
for _, er in etyma[a_mask].iterrows():
    eid = er["id"]; hg = er["homograph_number"]
    n_links = int((etyma_reflex["etyma_id"] == eid).sum())
    n_dist  = int(etyma_reflex.loc[etyma_reflex["etyma_id"]==eid,"reflex_id"].nunique())
    print(f"  IELex 'ā' hg={hg}: {n_dist} distinct reflexes ({n_links} links)")

# DPP ā has no ordinal prefix → root_filled = "ā", ord_key = SENTINEL
dpp_a = dpp_valid[dpp_valid["root_filled"].fillna("").str.strip() == "ā"]
print(f"  DPP 'ā':  {len(dpp_a)} reflex rows, {dpp_a['reflex'].nunique()} distinct")
print("  DPP abbrs:", dpp_a["abbr"].value_counts().to_dict())

print()
print("=== Sanity check: ad- (homophones 1 and 2) ===")
for hg_target in ["1","2"]:
    ad_mask = (etyma["entry"] == "ad-") & (etyma["homograph_number"].astype(str).str.startswith(hg_target))
    for _, er in etyma[ad_mask].iterrows():
        eid = er["id"]; hg = er["homograph_number"]
        n_dist = int(etyma_reflex.loc[etyma_reflex["etyma_id"]==eid,"reflex_id"].nunique())
        print(f"  IELex 'ad-' hg={hg}: {n_dist} distinct reflexes")

for dpp_root_str in ["1. ad-","2. ad-"]:
    dpp_ad = dpp_valid[dpp_valid["root_filled"].fillna("").str.strip() == dpp_root_str]
    print(f"  DPP '{dpp_root_str}': {len(dpp_ad)} reflex rows, {dpp_ad['reflex'].nunique()} distinct")
print()
print("Note: ā is under-populated (DPP 13 rows > IELex 7).")
print("Note: ad- homophones show IELex ≥ DPP — consistent with 43% of roots in that category.") 

=== Sanity check: ā ===
  IELex 'ā' hg=nan: 7 distinct reflexes (7 links)
  DPP 'ā':  13 reflex rows, 8 distinct
  DPP abbrs: {'gr.': 4, 'lit.': 3, 'lat.': 2, 'ai.': 1, 'got.': 1, 'ahd.': 1, 'mhd.': 1}

=== Sanity check: ad- (homophones 1 and 2) ===
  IELex 'ad-' hg=1: 41 distinct reflexes
  IELex 'ad-' hg=2: 23 distinct reflexes
  DPP '1. ad-': 26 reflex rows, 22 distinct
  DPP '2. ad-': 16 reflex rows, 13 distinct

Note: ā is under-populated (DPP 13 rows > IELex 7).
Note: ad- homophones show IELex ≥ DPP — consistent with 43% of roots in that category.


## Q-F — Column-level crosswalk: data profiling

Profiles **every column** of all 15 IELex `lex_*` tables, the DPP CSV, and the Starling
PIET JSON, then verifies the cross-source joins that underpin the crosswalk note
(`/LinguisticResearchCenter/IELEX/Column Crosswalk — DPP × IELex × PIET.md`). The point is
to demonstrate understanding of each field's *contents* from the data, not assert it.

### F.1 — Profile all 15 IELex tables (fill %, cardinality, samples)

In [81]:
import csv, glob, json as _json
csv.field_size_limit(10**7)

def profile_csv(path, max_samples=3, sample_len=55):
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    n = len(rows)
    print(f"\n{'='*72}\n{Path(path).name}   ({n:,} rows)\n{'='*72}")
    for c in (rows[0].keys() if rows else []):
        nb = [r[c] for r in rows if r[c] is not None and str(r[c]).strip() not in ("", "NULL")]
        seen = []
        for v in nb:
            s = " ".join(str(v).split())
            if s not in seen: seen.append(s)
            if len(seen) >= max_samples: break
        print(f"  {c:24} fill={len(nb)/n*100:5.1f}%  card={len(set(nb)):<6} "
              f"e.g. " + " | ".join(s[:sample_len] for s in seen))

for f in sorted(glob.glob(str(IELEX_DIR / "*.csv"))):
    profile_csv(f)


lex_etyma.csv   (2,222 rows)
  id                       fill=100.0%  card=2222   e.g. 3939 | 3943 | 3989
  old_id                   fill=100.0%  card=2222   e.g. 1717 | 1721 | 1767
  order                    fill=100.0%  card=2222   e.g. 17170 | 17210 | 17670
  page_number              fill=100.0%  card=1430   e.g. 917 | 918-19 | 958
  entry                    fill=100.0%  card=2016   e.g. (s)k(h)ai-, (s)k(h)ai-d-, (s)k(h)ai-t- | (s)k(h)ed-, (s)k(h)e-n-d- | (s)k<sup>u̯</sup>alo-s
  homograph_number         fill= 36.5%  card=27     e.g. 5 | 1 | 2
  gloss                    fill=100.0%  card=2012   e.g. {"en":"to cuff, kick"} | {"en":"to crush, <b>shatter<\/b>, split to pieces"} | {"en":"(type of large fish, e.g. <b>whale<\/b>)"}
  created_at               fill=100.0%  card=989    e.g. 2015-04-16 07:27:09 | 2015-04-16 07:27:15 | 2015-04-16 07:28:17
  updated_at               fill=100.0%  card=989    e.g. 2015-04-16 07:27:09 | 2015-04-16 07:27:15 | 2015-04-16 07:28:17
  lexicon_id       

### F.2 — Notable IELex column semantics (the non-obvious ones)

- `lex_reflex.lang_attribute` is an **ISO-639 code** (en, la, grc, ang, goh…) and is
  **functionally determined by `language_id`** (0/200 languages carry >1) — a denormalised
  per-reflex language tag, not independent data.
- `lex_etyma.homograph_number` is **not pure-numeric**: it includes alpha sub-variants
  (`2a–2e`, `3a–3i`). DPP's in-string ordinal is digits only, so DPP `3` may correspond to
  IELex `3` **and** `3a, 3b…`.
- `lex_reflex_part_of_speech.text` is **semi-structured free text**: only 67% of rows (42 of
  501 distinct values) match a `lex_part_of_speech.code`; the rest are composed tags
  (`n.masc`, `3.sg.past`, `abl.pl`). So `lex_part_of_speech` is a *partial* vocabulary,
  **not** a strict FK.
- `lex_etyma.entry` is **not unique** (140 strings reused by homographs, max 10); the natural
  key is `(entry, homograph_number)`.
- `lex_language.description` is `{"en":null}` for all 379 (the landing-page gap).

In [82]:
reflex   = list(csv.DictReader(open(IELEX_DIR / "lex_reflex.csv")))
etyma_l  = list(csv.DictReader(open(IELEX_DIR / "lex_etyma.csv")))
rpos     = list(csv.DictReader(open(IELEX_DIR / "lex_reflex_part_of_speech.csv")))
pos_codes = {p["code"] for p in csv.DictReader(open(IELEX_DIR / "lex_part_of_speech.csv"))}

import collections
bylang = collections.defaultdict(set)
for r in reflex: bylang[r["language_id"]].add(r["lang_attribute"])
print("languages with >1 distinct lang_attribute:",
      sum(1 for v in bylang.values() if len(v) > 1), "of", len(bylang))

hn = collections.Counter(e["homograph_number"] for e in etyma_l
                         if e["homograph_number"] not in ("", "NULL"))
print("homograph_number non-numeric values:", [v for v in hn if not v.isdigit()])

in_codes = sum(1 for r in rpos if r["text"] in pos_codes)
print(f"reflex POS rows whose text is a lex_part_of_speech.code: "
      f"{in_codes:,}/{len(rpos):,} ({in_codes/len(rpos)*100:.1f}%)")

languages with >1 distinct lang_attribute: 0 of 200
homograph_number non-numeric values: ['3a', '2b', '2a', '3b', '2d', '2e', '2c', '3c', '3d', '3e', '3f', '3g', '3h', '3i']
reflex POS rows whose text is a lex_part_of_speech.code: 43,341/64,315 (67.4%)


### F.3 — Profile DPP columns + index semantics

In [83]:
dpp_rows = list(csv.DictReader(open(DPP_FILE, encoding="utf-8-sig")))
n = len(dpp_rows)
print(f"DPP: {n:,} rows, {len(dpp_rows[0])} cols")
for c in dpp_rows[0]:
    nb = [r[c] for r in dpp_rows if r[c] and r[c].strip()]
    seen = []
    for v in nb:
        s = " ".join(v.split())
        if s not in seen: seen.append(s)
        if len(seen) >= 3: break
    print(f"  {c:26} fill={len(nb)/n*100:5.1f}%  card={len(set(nb)):<6} "
          + " | ".join(s[:42] for s in seen))

# index semantics: pok_all_index = per-root entry id (constant within a root);
# pok_wd_index = per-reflex running id (unique per row)
byroot = collections.defaultdict(set)
for r in dpp_rows: byroot[r["root"]].add(r["pok_all_index"])
const = sum(1 for v in byroot.values() if len(v) == 1)
print(f"\npok_all_index constant within a root: {const}/{len(byroot)} roots "
      f"(=> it is a per-ENTRY id)")
print("pok_wd_index unique per row:",
      len({r['pok_wd_index'] for r in dpp_rows}) == n, "(=> per-REFLEX id)")

DPP: 67,656 rows, 13 cols
  pok_all_index              fill=100.0%  card=2211   0 | 1 | 2
  pok_wd_index               fill=100.0%  card=67656  0 | 1 | 2
  web_root                   fill=100.0%  card=2209   ā | ab- | ā̆bel-, ā̆bōl-, ab↓↓e↓↓l-
  root                       fill= 89.5%  card=1517   ā | ab- | ā̆bel-, ā̆bō̆l-, abel-
  abbr                       fill= 98.9%  card=1789   ai. | gr. | lat.
  reflex                     fill= 99.0%  card=63122  ā | ἆ | ἀά
  meaning                    fill= 94.1%  card=32656  Ausruf der Besinnung | Ausruf des Unwillens, Schmerzes, Erstaunen | Ausruf der Verwunderung und Klage
  notes (specific)           fill= 61.1%  card=24415  lauter Neuschöpfungen | auch dem Vokativ angehängt | f., später m. c(*abnis)
  notes (general/overflow)   fill= 44.0%  card=3141   Im Kelt. sind die Bezeichnungen für ‘Apfel | Die gleichen Ablautformen im Germanischen: | Die gleichen Ablautformen im Germanischen:
  notes (shared/overflow)    fill=  4.6%  card=291    Germ.

### F.4 — Profile Starling PIET structure (top-level + sub-databases)

In [84]:
piet = _json.load(open(REPO / "data/ielex/starling_piet/piet_complete.json"))
print(f"PIET records: {len(piet):,}")

topkeys = collections.Counter()
for r in piet: topkeys.update(r.keys())
print("\nTop-level fields (Starling IE etymology layer):")
for k, v in topkeys.most_common():
    print(f"  {k:22} in {v:,} recs")

base = collections.Counter()
subkeys = collections.defaultdict(collections.Counter)
for r in piet:
    for s in r.get("_sub_entries", []):
        b = s.get("_basename", "?"); base[b] += 1; subkeys[b].update(s.keys())
print("\nSub-database (_basename) frequency  -- Starling is NOT only Pokorny:")
print(" ", dict(base))
print("\n'pokorny' sub-entry fields (the IELex-mappable ones):")
for k, v in subkeys["pokorny"].most_common():
    print(f"    {k:22} {v:,}")

PIET records: 3,178

Top-level fields (Starling IE etymology layer):
  _page                  in 3,178 recs
  _record_num            in 3,178 recs
  _sub_entries           in 3,178 recs
  Proto-IE               in 3,178 recs
  Meaning                in 3,178 recs
  _content_hash          in 3,178 recs
  Russ. meaning          in 3,144 recs
  References             in 3,057 recs
  Germanic               in 1,983 recs
  Old Greek              in 1,698 recs
  Baltic                 in 1,615 recs
  Latin                  in 1,391 recs
  Slavic                 in 1,368 recs
  Old Indian             in 1,330 recs
  Celtic                 in 1,028 recs
  Nostratic etymology    in 757 recs
  Avestan                in 674 recs
  Armenian               in 504 recs
  Tokharian              in 486 recs
  Comments               in 381 recs
  Hittite                in 330 recs
  Other Iranian          in 322 recs
  Other Italic           in 254 recs
  Albanian               in 235 recs

Sub-database

### F.5 — Cross-source join verification (underpins the crosswalk)

In [85]:
import re as _re
first = lambda s: (m.group() if (m := _re.search(r"\d+", str(s or ""))) else None)

# (1) PIET pokorny 'Pages' -> IELex page_number (first page)
iel_pages = {first(e["page_number"]) for e in etyma_l} - {None}
pk_pages = [first(s.get("Pages")) for r in piet for s in r.get("_sub_entries", [])
            if s.get("_basename") == "pokorny"]
pk_pages = [p for p in pk_pages if p]
hit = sum(1 for p in pk_pages if p in iel_pages)
print(f"PIET pokorny Pages -> IELex page_number : {hit}/{len(pk_pages)} "
      f"({hit/len(pk_pages)*100:.1f}%)")

# (2) reflex-count parity DPP vs IELex
dpp_reflexes = sum(1 for r in dpp_rows if r["reflex"].strip())
print(f"DPP reflex rows: {dpp_reflexes:,}  vs  IELex lex_reflex: {len(reflex):,}  (near-parity)")

# (3) laryngeal upgrade? (h1/h2/h3) in PIET pokorny Root
lar = lambda s: bool(_re.search(r"h[\u2081\u2082\u2083123]", str(s or "")))
pkroot = [s.get("Root", "") for r in piet for s in r.get("_sub_entries", [])
          if s.get("_basename") == "pokorny"]
print(f"PIET pokorny Roots with numbered laryngeals: {sum(lar(x) for x in pkroot)}/{len(pkroot)} "
      f"(=> classic Pokorny, no free upgrade)")

PIET pokorny Pages -> IELex page_number : 1680/1681 (99.9%)
DPP reflex rows: 66,994  vs  IELex lex_reflex: 63,116  (near-parity)
PIET pokorny Roots with numbered laryngeals: 0/1681 (=> classic Pokorny, no free upgrade)


## Q-G — Reflex sourcing: Pokorny vs other dictionaries (apples-to-apples) + tiering

Added 2026-06-07 after Todd's question: **when IELex out-counts Starling/DPP, are those extra
reflexes from Pokorny itself, or from other dictionaries (Webster's, AHD, …)?**

Every IELex reflex carries its cited source in `lex_reflex_source` → `lex_source` (44 dictionaries).
This section distinguishes two things that are easy to conflate — *citing* Pokorny (G.1) vs a form
*actually being in* Pokorny (G.2/G.2b) — and shows the source field is the lever for Todd's
Tier-1/Tier-2 plan.

> **Bottom line (corrected 2026-06-07).** Two different measures were conflated in an earlier draft:
> **citing** Pokorny (G.1 — editorial bookkeeping; only 1.7% of reflexes credit IEW) vs a form
> **actually appearing in** Pokorny (G.2/G.2b — string match, the real measure). Pokorny is **not**
> a 1,100-reflex dataset: DPP has ~67k reflex rows (~44/root), comparable to IELex's 63k. On the
> 1,357 shared roots the two sets are similar in size but overlap only ~25–30%: ~40k IELex-only
> (Webster's/AHD English — the *psychology* layer) vs ~38k Pokorny-only (older daughter-language
> forms we lack). So the gap is real and **two-directional**, and it rests on the string-match,
> NOT the citation count.

### G.1 — Which reference books the LRC *cited* (`lex_reflex_source` → `lex_source`)

⚠️ **Citation practice, not a Pokorny-content measure.** The source field records which book the
LRC editor credited for each form — overwhelmingly working desk dictionaries (Webster's, AHD,
Lehmann), and rarely Pokorny directly even for forms straight out of Pokorny. So "1.7% cite IEW"
does **not** mean "1.7% of our reflexes are in Pokorny" (that is G.2). It only shows what the LRC
built from day-to-day.

In [ ]:
# Q-G.1 — Which reference book the LRC CITED per reflex. lex_reflex_source (76,678 links) -> lex_source.
# NOTE: this is citation practice (the book the editor credited), NOT a measure of Pokorny content.
# The content measure is G.2 (string-match); the two differ ~10x precisely because citation != content.
import csv
from collections import Counter, defaultdict

src_meta = {r["id"]: r for r in csv.DictReader(open(IELEX_DIR / "lex_source.csv"))}

ref_src = defaultdict(set)                       # reflex_id -> {source_id, ...}
for r in csv.DictReader(open(IELEX_DIR / "lex_reflex_source.csv")):
    ref_src[r["reflex_id"]].add(r["source_id"])

all_reflex_ids = [r["id"] for r in csv.DictReader(open(IELEX_DIR / "lex_reflex.csv"))]
n_ref  = len(all_reflex_ids)
IEW_ID = "70"                                    # Julius Pokorny: IEW (1959)

per_source = Counter()
for rid in all_reflex_ids:
    for s in ref_src.get(rid, ()):
        per_source[s] += 1                       # distinct reflexes per source (a reflex may cite several)

g1 = pd.DataFrame(
    [(src_meta[s]["code"], src_meta[s]["display"][:48], c, round(100 * c / n_ref, 1))
     for s, c in per_source.most_common(8)],
    columns=["code", "source", "reflexes", "pct_of_63116"],
)
print("=== G.1 reflexes by source (top 8) ===")
print(g1.to_string(index=False))
print(f"\nReflexes citing Pokorny/IEW at all: {per_source[IEW_ID]} "
      f"({100 * per_source[IEW_ID] / n_ref:.1f}%)")

=== G.1 reflexes by source (top 8) ===
code                                           source  reflexes  pct_of_63116
  W7 Webster's Seventh New Collegiate Dictionary (196     33888          53.7
 AHD Calvert Watkins: The American Heritage Dictionar     13114          20.8
 ASD Joseph Bosworth and T. Northcote Toller: An Angl      7457          11.8
 LRC Linguistics Research Center, University of Texas      6521          10.3
 GED Winfred P. Lehmann: A Gothic Etymological Dictio      4871           7.7
 CDC W.D. Whitney and B.E. Smith: The Century Diction      2883           4.6
 RPN Allan R. Bomhard: Reconstructing Proto-Nostratic      2684           4.3
 TLL    Frederick Bodmer: The Loom of Language (1944)      2260           3.6

Reflexes citing Pokorny/IEW at all: 1100 (1.7%)


### G.2 — The real measure: is the reflex *form* actually in DPP/Pokorny?

The source field says *where a form was attested*, not "in Pokorny / not" — a form cited to
Webster's could still be in Pokorny. So restrict to the Q-B matched roots and split IELex's
reflexes by whether the **string itself** appears in DPP's Pokorny (same homograph-aware key as Q-B).
This is the trustworthy overlap measure (validated in G.2b: 92% of the forms the LRC *did* credit
to Pokorny string-match DPP).

In [ ]:
# Q-G.2 — Rigorous apples-to-apples.  Reuses src_meta / ref_src from G.1 and
# norm_strip / ordinal_of from the setup cell.  Join key matches Q-B:
#   (norm_strip(root), ordinal) <-> (norm_strip(entry), homograph_number).
import csv, json, re, unicodedata
from collections import Counter, defaultdict

def _hgnum(h):                                   # IELex homograph_number -> leading digits ('' if none)
    if not h or h == "NULL":
        return ""
    m = re.match(r"(\d+)", h)
    return m.group(1) if m else ""

def _nf(s):                                      # reflex-string normalizer: strip diacritics, keep letters
    if not s:
        return ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-zͰ-Ͽ]+", "", s.lower())   # ASCII + Greek block

# DPP: (stripped_root, ordinal) -> set of normalized reflex strings
dpp_ref_strings = defaultdict(set)
for r in csv.DictReader(open(DPP_FILE, encoding="utf-8-sig")):
    root = r["root"]
    if not root.strip():
        continue
    v = _nf(r["reflex"])
    if v:
        dpp_ref_strings[(norm_strip(root), ordinal_of(root))].add(v)

# IELex: etyma join key + per-reflex normalized surface strings
et_key = {e["id"]: (norm_strip(e["entry"]), _hgnum(e["homograph_number"]))
          for e in csv.DictReader(open(IELEX_DIR / "lex_etyma.csv"))}
et_links = defaultdict(list)
for l in csv.DictReader(open(IELEX_DIR / "lex_etyma_reflex.csv")):
    et_links[l["etyma_id"]].append(l["reflex_id"])
reflex_strings = {}
for r in csv.DictReader(open(IELEX_DIR / "lex_reflex.csv")):
    try:
        forms = [x.get("text", "") for x in json.loads(r["entries"])]
    except Exception:
        forms = [r["entries"]]
    reflex_strings[r["id"]] = {_nf(x) for x in forms if x}

ielex_by_key = defaultdict(list)
for eid, key in et_key.items():
    for rid in et_links.get(eid, []):
        ielex_by_key[key].append(rid)
matched_keys = [k for k in ielex_by_key if k in dpp_ref_strings]

surplus_src, n_in_dpp, n_only = Counter(), 0, 0
for k in matched_keys:
    dpp_set = dpp_ref_strings[k]
    for rid in ielex_by_key[k]:
        if reflex_strings.get(rid, set()) & dpp_set:
            n_in_dpp += 1                        # genuine Pokorny overlap
        else:
            n_only += 1                          # IELex-only surplus
            for s in ref_src.get(rid, ()):
                surplus_src[s] += 1

print(f"=== G.2 apples-to-apples on {len(matched_keys)} matched roots ===")
print(f"IELex reflexes matching DPP (Pokorny overlap): {n_in_dpp}")
print(f"IELex-only (not in DPP) surplus              : {n_only}")
g2 = pd.DataFrame(
    [(src_meta[s]["code"], src_meta[s]["display"][:48], c, round(100 * c / n_only, 1))
     for s, c in surplus_src.most_common(7)],
    columns=["code", "source", "reflexes", "pct_of_surplus"],
)
print(g2.to_string(index=False))
print("\nCaveat: string-match across German/IELex orthography is imperfect -> 'IELex-only' is a"
      "\nslight over-count, but the Webster's/AHD dominance is far too large to be spelling noise.")

=== G.2 apples-to-apples on 1357 matched roots ===
IELex reflexes matching DPP (Pokorny overlap): 15867
IELex-only (not in DPP) surplus              : 40404
code                                           source  reflexes  pct_of_surplus
  W7 Webster's Seventh New Collegiate Dictionary (196     25328            62.7
 AHD Calvert Watkins: The American Heritage Dictionar     10902            27.0
 ASD Joseph Bosworth and T. Northcote Toller: An Angl      3522             8.7
 LRC Linguistics Research Center, University of Texas      3052             7.6
 CDC W.D. Whitney and B.E. Smith: The Century Diction      2058             5.1
 GED Winfred P. Lehmann: A Gothic Etymological Dictio      1734             4.3
 TLL    Frederick Bodmer: The Loom of Language (1944)      1656             4.1

Caveat: string-match across German/IELex orthography is imperfect -> 'IELex-only' is a
slight over-count, but the Webster's/AHD dominance is far too large to be spelling noise.


### G.2b — Resolving "why is overlap (G.2) 10× the citation count (G.1)?"

Citation ≠ content. Here: (a) Pokorny's true scale (it is *not* a ~1,100-reflex dataset),
(b) a validation that the string-matcher is reliable, (c) the two-directional overlap that frames
the merge plan.

In [ ]:
# Q-G.2b — citation vs content; Pokorny scale; matcher validation; two-directional overlap.
# Reuses DPP_FILE, n_ref, ref_src, IEW_ID (G.1) and dpp_ref_strings, ielex_by_key,
# reflex_strings, matched_keys (G.2).
import csv
from collections import defaultdict

# (a) Pokorny's true scale — NOT a ~1,100-reflex dataset
dpp_rows = [r for r in csv.DictReader(open(DPP_FILE, encoding="utf-8-sig")) if r["reflex"].strip()]
dpp_roots = {r["root"].strip() for r in dpp_rows if r["root"].strip()}
print(f"Pokorny/DPP: {len(dpp_rows):,} reflex rows over {len(dpp_roots):,} roots "
      f"= {len(dpp_rows)/len(dpp_roots):.1f} reflexes/root (cf. IELex's {n_ref:,})")

# (b) validate the matcher: of reflexes the LRC DID credit to Pokorny, how many appear in DPP?
rid2key = {rid: k for k in matched_keys for rid in ielex_by_key[k]}
hit = miss = 0
for rid, ss in ref_src.items():
    if IEW_ID not in ss:
        continue
    k = rid2key.get(rid)
    if not k:
        continue
    if reflex_strings.get(rid, set()) & dpp_ref_strings[k]:
        hit += 1
    else:
        miss += 1
print(f"IEW-cited reflexes on a matched root: {hit} appear in DPP, {miss} do not "
      f"({100*hit/(hit+miss):.0f}% confirm) -> citation is a reliable SUBSET of content")

# (c) two-directional overlap on matched roots (distinct normalized forms)
inter = ie_only = dpp_only = 0
for k in matched_keys:
    a = set().union(*(reflex_strings.get(rid, set()) for rid in ielex_by_key[k])) - {""}
    b = dpp_ref_strings[k] - {""}
    inter += len(a & b); ie_only += len(a - b); dpp_only += len(b - a)
print(f"\nDistinct reflex forms on {len(matched_keys)} shared roots:")
print(f"  in BOTH                                      : {inter:,}")
print(f"  IELex-only (Webster's/AHD English layer)     : {ie_only:,}")
print(f"  Pokorny-only (older daughter langs we lack)  : {dpp_only:,}")

Pokorny/DPP: 66,994 reflex rows over 1,509 roots = 44.4 reflexes/root (cf. IELex's 63,116)
IEW-cited reflexes on a matched root: 905 appear in DPP, 80 do not (92% confirm) -> citation is a reliable SUBSET of content

Distinct reflex forms on 1357 shared roots:
  in BOTH                                      : 11,912
  IELex-only (Webster's/AHD English layer)     : 40,031
  Pokorny-only (older daughter langs we lack)  : 38,496


### G.3 — Implication for Todd's tiering plan

Tiering needs **no new data collection** — classify the existing 44 `lex_source` rows:

| Tier | Sources (examples) | Rationale |
|------|--------------------|-----------|
| **Tier 1** | IEW (Pokorny), Lehmann GED, Liddell-Scott, Köbler series, EIE/Mallory-Adams, classical etymological dicts | "Fundamental" comparative etymologies — academically load-bearing |
| **Tier 2** | Webster's 7th/9th/2nd, AHD (Watkins), OED, ODE, Century Dict., bilingual desk dictionaries | Modern-English derivatives (e.g. *psychology*): useful for non-specialists, no new academic info — keep but de-clutter |

This is the lever to keep *psychology* visible without burying the fundamental etymologies.
Detail in the appendix of `LinguisticResearchCenter/IELEX/Reply to colleague — Pokorny gaps & Starling — 2026-06-06.md`.